# tiny-kws — full training run (free Colab T4)

Trains the DS-CNN keyword spotter for 30 epochs on Google Speech Commands v2 (12-class benchmark).

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Expected wall time: ~5 min setup + data prep, ~20-30 min training. Checkpoints are saved to your Google Drive every epoch, so a disconnect costs nothing — just rerun the training cell with `--epochs` reduced by however many completed, or simply restart it (it reuses the prepared data).

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/priyadeepjaiswal9c/tiny-kws.git
%cd tiny-kws
!pip install -q -r requirements.txt

In [ ]:
# Download Speech Commands v2 (2.4 GB) + the official curated test set.
# Colab pulls from Google storage at ~100 MB/s, so this takes ~1 minute.
!mkdir -p data/SpeechCommands/speech_commands_v0.02 data/test_set
!curl -sSL -o data/speech_commands_v0.02.tar.gz https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz
!curl -sSL -o data/speech_commands_test_set_v0.02.tar.gz https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_test_set_v0.02.tar.gz
!tar -xzf data/speech_commands_v0.02.tar.gz -C data/SpeechCommands/speech_commands_v0.02
!tar -xzf data/speech_commands_test_set_v0.02.tar.gz -C data/test_set --strip-components=0
# the test tarball extracts into a nested dir on some tar versions — flatten if needed
import os, shutil
nested = 'data/test_set/speech_commands_test_set_v0.02'
if os.path.isdir(nested):
    for d in os.listdir(nested):
        shutil.move(os.path.join(nested, d), 'data/test_set/')
    os.rmdir(nested)
!rm data/*.tar.gz
!ls data/SpeechCommands/speech_commands_v0.02 | head

In [ ]:
# Build the 12-class splits (official lists) and feature caches (~5 min)
!python src/prepare_data.py

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/tiny-kws/checkpoints'
!mkdir -p {CKPT_DIR}

In [ ]:
# Full training run: 30 epochs on the T4 (~20-30 min).
# best.pt / last.pt / history.json land in your Drive.
!python src/train.py --epochs 30 --out-dir {CKPT_DIR}

In [ ]:
# Honest final numbers: official test set only
!python src/evaluate.py --ckpt {CKPT_DIR}/best.pt
!cp assets/metrics.json assets/confusion_matrix.png {CKPT_DIR}/

In [ ]:
# Also download best.pt directly to your machine (it is already in Drive too)
from google.colab import files
files.download(f'{CKPT_DIR}/best.pt')
files.download(f'{CKPT_DIR}/metrics.json')
files.download(f'{CKPT_DIR}/confusion_matrix.png')

## Done

You now have, in `Drive/tiny-kws/checkpoints/`:
- `best.pt` — checkpoint with the best validation accuracy (weights + config + label order + feature stats)
- `metrics.json` + `confusion_matrix.png` — test-set evaluation artifacts
- `history.json` — per-epoch training curves

Drop `best.pt` into the local repo's `checkpoints/` directory to deploy it.